# Esteira de Aprendizado de Máquina - Base Plants (UCI)

**Disciplina:** Inteligência Artificial
**Base:** Plants - UCI Machine Learning Repository (id 180) - https://archive.ics.uci.edu/dataset/180/plants

## Sobre a base
A base vem do banco de dados USDA Plants e está em formato **transacional**: cada linha tem o nome latino de uma planta (espécie ou gênero) seguido da lista de estados dos EUA e províncias do Canadá onde ela ocorre. Na página da UCI, a tarefa associada é clusterização, então aqui eu adaptei a base para um problema de **classificação**.

## Problema de classificação
> Dada a lista de estados dos EUA em que uma espécie de planta ocorre, prever se ela **também ocorre no Canadá** (classe 1) ou **não** (classe 0).

## Etapas da esteira
1. Download e leitura da base
2. Estatísticas descritivas
3. Transformações nas linhas
4. Criação da variável alvo e transformações nas colunas
5. Divisão em treino, validação e teste
6. Treinamento e escolha do modelo (validação)
7. Avaliação final (teste): acurácia e matriz de confusão
8. Predição de exemplo

In [ ]:
import os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
pd.set_option("display.max_columns", 20)

## 1. Download e leitura da base

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/180/plants.zip"
os.makedirs("dados", exist_ok=True)

req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp, open("dados/plants.zip", "wb") as f:
    f.write(resp.read())

with zipfile.ZipFile("dados/plants.zip") as z:
    print("Arquivos no zip:", z.namelist())
    z.extractall("dados")

# Caso o download falhe, baixe o zip no site da UCI e envie os arquivos
# para a pasta "dados" do Colab (menu de arquivos à esquerda).

In [ ]:
# Abreviações de estados e províncias (arquivo de apoio da própria base)
print(open("dados/stateabbr.txt", encoding="latin-1").read())

In [ ]:
# Leitura do formato transacional: nome, estado1, estado2, ...
registros = []
with open("dados/plants.data", encoding="latin-1") as f:
    for linha in f:
        partes = [p.strip().lower() for p in linha.strip().split(",")]
        if partes and partes[0]:
            registros.append((partes[0], [p for p in partes[1:] if p]))

print("Total de linhas lidas:", len(registros))
for nome, estados in registros[:3]:
    print(nome, "->", estados)

In [ ]:
# Converte a lista de estados em colunas 0/1 (uma coluna por estado/província)
mlb = MultiLabelBinarizer()
matriz = mlb.fit_transform([estados for _, estados in registros])

df = pd.DataFrame(matriz, columns=mlb.classes_).astype("int8")
df.insert(0, "nome", [nome for nome, _ in registros])
df.head()

## 2. Estatísticas descritivas gerais

In [ ]:
print("Dimensões (linhas, colunas):", df.shape)
print("Valores nulos:", int(df.isna().sum().sum()))
print("Nomes duplicados:", int(df["nome"].duplicated().sum()))

colunas_local = [c for c in df.columns if c != "nome"]
df["n_locais"] = df[colunas_local].sum(axis=1)

print("\nEstatísticas do número de locais por planta:")
print(df["n_locais"].describe())
print("\nPlantas sem nenhum local registrado:", int((df["n_locais"] == 0).sum()))

In [ ]:
# Estatísticas descritivas de cada coluna de local (média = proporção de plantas que ocorrem lá)
df[colunas_local].describe().T.sort_values("mean", ascending=False).head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df[colunas_local].mean().sort_values(ascending=False).head(15).plot.bar(ax=axes[0])
axes[0].set_title("15 locais com mais plantas (proporção)")

df["n_locais"].plot.hist(bins=40, ax=axes[1])
axes[1].set_title("Distribuição do número de locais por planta")
axes[1].set_xlabel("Número de locais")

plt.tight_layout()
plt.show()

## 3. Transformações nas linhas
- **Remover linhas duplicadas** (mesmo nome de planta).
- **Remover linhas sem nenhum local** registrado (não trazem informação).
- **Remover linhas de gênero** (nomes com uma só palavra). Elas são agregados de várias espécies e misturariam níveis diferentes de informação na mesma base.

In [ ]:
antes = len(df)

df = df.drop_duplicates(subset="nome")
print("Após remover duplicadas:", len(df))

df = df[df["n_locais"] > 0]
print("Após remover linhas sem local:", len(df))

df = df[df["nome"].str.split().str.len() >= 2]
print("Após remover linhas de gênero:", len(df))

df = df.reset_index(drop=True)
print(f"\nLinhas removidas no total: {antes - len(df)}")

## 4. Variável alvo e transformações nas colunas
- **Criar a variável alvo** `ocorre_canada`: 1 se a planta ocorre em alguma província canadense, 0 caso contrário.
- **Remover as colunas das províncias canadenses** dos atributos, porque elas dão a resposta direta (vazamento de dados).
- **Criar a coluna** `n_estados` com o total de estados dos EUA onde a planta ocorre.
- **Remover a coluna** `nome` (texto) e a coluna temporária `n_locais`.

In [ ]:
# Códigos de províncias/territórios do Canadá - confira com o stateabbr.txt acima
PROVINCIAS = ["ab", "bc", "mb", "nb", "nf", "lb", "nt", "nu", "ns", "on", "pe", "qc", "sk", "yt"]
provincias_na_base = [c for c in PROVINCIAS if c in df.columns]
print("Províncias encontradas na base:", provincias_na_base)

# Variável alvo
df["ocorre_canada"] = (df[provincias_na_base].sum(axis=1) > 0).astype(int)

# Colunas: remover províncias, criar n_estados, remover colunas que não serão usadas
df = df.drop(columns=provincias_na_base)
colunas_estados = [c for c in df.columns if c not in ("nome", "n_locais", "ocorre_canada")]
df["n_estados"] = df[colunas_estados].sum(axis=1)
df = df.drop(columns=["n_locais"])

nomes = df["nome"]
X = df.drop(columns=["nome", "ocorre_canada"])
y = df["ocorre_canada"]

print("Formato de X:", X.shape)
print("\nDistribuição da classe alvo:")
print(y.value_counts())
print((y.value_counts(normalize=True) * 100).round(1).astype(str) + "%")

## 5. Divisão em treino, validação e teste
Divisão estratificada (mantém a proporção das classes): **70% treino, 15% validação, 15% teste**.

In [ ]:
X_treino, X_temp, y_treino, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

X_val, X_teste, y_val, y_teste = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print("Treino:   ", X_treino.shape)
print("Validação:", X_val.shape)
print("Teste:    ", X_teste.shape)

## 6. Treinamento e escolha do modelo
Modelo: **Random Forest**. Treino com diferentes profundidades máximas e escolha da melhor pela **acurácia na validação** (o conjunto de teste fica guardado para a avaliação final).

In [ ]:
resultados = []
for prof in [5, 10, 20, None]:
    modelo_tmp = RandomForestClassifier(
        n_estimators=200, max_depth=prof, random_state=SEED, n_jobs=-1)
    modelo_tmp.fit(X_treino, y_treino)
    acc_val = accuracy_score(y_val, modelo_tmp.predict(X_val))
    resultados.append((prof, acc_val))
    print(f"max_depth={prof}: acurácia na validação = {acc_val:.4f}")

melhor_prof = max(resultados, key=lambda r: r[1])[0]
print("\nMelhor max_depth:", melhor_prof)

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=200, max_depth=melhor_prof, random_state=SEED, n_jobs=-1)
modelo.fit(X_treino, y_treino)
print("Modelo treinado.")

## 7. Avaliação final no conjunto de teste

In [ ]:
y_pred = modelo.predict(X_teste)

acuracia = accuracy_score(y_teste, y_pred)
print(f"Acurácia no teste: {acuracia:.4f} ({acuracia*100:.2f}%)\n")
print(classification_report(y_teste, y_pred, target_names=["Não ocorre no Canadá", "Ocorre no Canadá"]))

In [ ]:
cm = confusion_matrix(y_teste, y_pred)
print("Matriz de confusão:\n", cm)

ConfusionMatrixDisplay(cm, display_labels=["Não ocorre\nno Canadá", "Ocorre\nno Canadá"]).plot(cmap="Blues")
plt.title("Matriz de confusão - conjunto de teste")
plt.show()

## 8. Predição de exemplo
Função que recebe a lista de estados onde uma planta ocorre e prevê se ela também ocorre no Canadá.

In [ ]:
def prever_planta(estados):
    estados = [e.strip().lower() for e in estados]
    linha = pd.DataFrame(0, index=[0], columns=X.columns)
    for e in estados:
        if e in linha.columns and e != "n_estados":
            linha.loc[0, e] = 1
        else:
            print(f"Aviso: '{e}' não é um estado conhecido da base, ignorado.")
    linha["n_estados"] = linha[colunas_estados].sum(axis=1)

    classe = modelo.predict(linha)[0]
    prob = modelo.predict_proba(linha)[0][1]
    texto = "OCORRE no Canadá" if classe == 1 else "NÃO ocorre no Canadá"
    print(f"Estados informados: {estados}")
    print(f"Predição: {texto} (probabilidade de ocorrer no Canadá: {prob:.1%})\n")

# Exemplo 1: planta de região quente do sul dos EUA
prever_planta(["fl", "ga", "al"])

# Exemplo 2: planta de região norte, com ampla distribuição
prever_planta(["wa", "or", "mt", "id", "wy", "nd", "mn", "wi", "mi", "ny", "vt", "me"])

In [ ]:
# Exemplo 3: uma planta real do conjunto de teste
i = X_teste.index[0]
print("Planta:", nomes.loc[i])
print("Valor real:", "ocorre no Canadá" if y_teste.loc[i] == 1 else "não ocorre no Canadá")
print("Predição do modelo:", "ocorre no Canadá" if modelo.predict(X_teste.loc[[i]])[0] == 1 else "não ocorre no Canadá")

## Conclusão
A esteira cobriu todas as etapas: leitura e adaptação da base transacional, estatísticas descritivas, transformações em linhas e colunas, divisão em treino/validação/teste, treinamento, avaliação com acurácia e matriz de confusão, e predição de exemplo.

**Observação:** a base original é voltada para clusterização, então a tarefa de classificação foi definida por mim a partir dos dados (ocorrência no Canadá).